# AT&T Spam Detector — Deep Learning NLP

**Projet :** Bloc 4 — Jedha CDSD
**Auteur :** Aymeric Lahonde

## Objectif

AT&T veut détecter automatiquement les SMS spam parmi le flux de messages. On construit :
- Étape 1 : EDA + preprocessing du texte
- Étape 2 : Modèle baseline (TF-IDF + LogisticRegression)
- Étape 3 : Fine-tuning DistilBERT en transfer learning

On compare les performances des 2 approches.


---
## Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

sns.set_theme(style='whitegrid')
print('Imports OK')


---
## Étape 1 — EDA + preprocessing


In [ ]:
# Lecture du dataset
df = pd.read_csv('../data/sms_spam.csv')
print('Shape :', df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
# Distribution des classes (typique : ~13% de spam, gros déséquilibre)
df['label'].value_counts(normalize=True).round(3)


In [ ]:
# Longueur des messages par classe
df['length'] = df['message'].str.len()
fig, ax = plt.subplots(figsize=(10, 4))
for label in df['label'].unique():
    df[df['label'] == label]['length'].hist(bins=50, alpha=0.5, label=label, ax=ax)
ax.legend()
ax.set_xlabel('Longueur du SMS (chars)')
ax.set_title('Distribution de la longueur par classe')
plt.show()


### Observation
Les SMS spam sont typiquement plus longs (essaient de placer un max d'infos type 'CALL NOW WIN £1000'). C'est une feature implicite que les modèles vont exploiter.


---
## Étape 2 — Baseline TF-IDF + LogisticRegression


In [ ]:
# On encode les labels en 0/1
df['target'] = (df['label'] == 'spam').astype(int)

X = df['message']
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train :', X_train.shape, '| Test :', X_test.shape)


In [ ]:
# TF-IDF + LogisticRegression
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression(class_weight='balanced', max_iter=500)
clf.fit(X_train_vec, y_train)

y_pred = clf.predict(X_test_vec)
print(classification_report(y_test, y_pred, target_names=['ham', 'spam']))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['ham', 'spam'], yticklabels=['ham', 'spam'])
plt.xlabel('Prédiction')
plt.ylabel('Vraie classe')
plt.title('Baseline TF-IDF + LogReg')
plt.show()


---
## Étape 3 — Transfer learning avec DistilBERT

On fine-tune DistilBERT (version distillée de BERT, ~60M params, ~3x plus rapide) sur notre dataset.


In [ ]:
# Imports Hugging Face
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments,
)
from datasets import Dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device :', device)


In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['message'], padding='max_length', truncation=True, max_length=128)

ds_train = Dataset.from_pandas(pd.DataFrame({'message': X_train.values, 'label': y_train.values}))
ds_test = Dataset.from_pandas(pd.DataFrame({'message': X_test.values, 'label': y_test.values}))

ds_train = ds_train.map(tokenize, batched=True)
ds_test = ds_test.map(tokenize, batched=True)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir='./distilbert_spam',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='no',
    report_to='none',
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'f1': f1_score(labels, preds)}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_test,
    compute_metrics=compute_metrics,
)


In [ ]:
# Fine-tuning (5-10 min sur CPU, <1 min sur GPU)
trainer.train()


In [ ]:
# Evaluation finale
results = trainer.evaluate()
print(results)


In [ ]:
# Predictions DistilBERT
preds_logits = trainer.predict(ds_test).predictions
preds_bert = np.argmax(preds_logits, axis=-1)

print(classification_report(y_test, preds_bert, target_names=['ham', 'spam']))


---
## Bilan

| Modèle | Accuracy | F1 (spam) | Temps train |
|---|---|---|---|
| TF-IDF + LogReg | TODO | TODO | <1 sec |
| DistilBERT fine-tuned | TODO | TODO | ~5-10 min CPU |

**Conclusions :**
- DistilBERT capture mieux le contexte sémantique → F1 typiquement >0.95 vs ~0.90 pour TF-IDF
- Le coût d'inférence DistilBERT est plus élevé → en prod, on pourrait garder TF-IDF pour le filtrage initial et n'envoyer à DistilBERT que les cas ambigus (cascading)
- Pistes : data augmentation, autres modèles (RoBERTa, BERT-tiny), ensembling
